In [8]:
from filterpy.kalman import KalmanFilter
from project_brain_decoder.train_gru import data_folder
from src.project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
import numpy as np
import gc

In [9]:
f = KalmanFilter(dim_x=2, dim_z=192)
f.x = np.array([[0.], [0.]])
f.P = 1000 * np.eye(2)
f.F = ... # learned from data
f.H = ... # learned from data
f.Q = ... # learned from data
f.R = ... # learned from data

In [10]:
files = list(data_folder.glob("*.nwb"))
train_files = files[:10]
neural_list = []
targets_list = []
X_list = []
y_list = []
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [ ]:
for file in train_files:
    loaded_file = load_nwb(file_path=file)
    spiking = loaded_file["neural_spiking_band"]
    threshold = loaded_file["neural_threshold_crossings"]
    neural = np.concatenate([spiking, threshold], axis=1)
    index = loaded_file["target_index_velocity"]
    mrs = loaded_file["target_mrs_velocity"]
    targets = np.column_stack([index, mrs])
    neural_list.append(neural)
    targets_list.append(targets)
neural_all = np.concatenate(neural_list, axis=0)
targets_all = np.concatenate(targets_list, axis=0)
neural_scaler.fit(neural_all)
targets_scaler.fit(targets_all)
del neural_all, targets_all
gc.collect()

In [ ]:
for neural, targets in zip(neural_list, targets_list):
    scaled_n = neural_scaler.transform(neural)
    scaled_t = targets_scaler.transform(targets)
    X_list.append(scaled_n)
    y_list.append(scaled_t)
X_train = np.concatenate(X_list)
y_train = np.concatenate(y_list)

In [5]:
# T = 27649
# for t in range(T):
#     f.predict()
#     f.update(z[t])
#     predictions.append(f.x)